# 第6回 演習：解釈と報告（解答例・教員用）

## 今日の分析目標

**何が利用台数を動かすのか、根拠をもって説明したい。**

この演習では、学習済みモデルから標準化係数を取り出して影響の向きと大きさを読み、並べ替え重要度で裏を取り、そして双子の変数（気温 temp と体感温度 atemp）が引き起こす多重共線性を実験で暴きます。係数の取り出しから日本語への翻訳、重要度による裏取り、そして多重共線性の実験までを自分の手で動かします。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
feature_names = ['temp','atemp','hum','windspeed','season','yr',
                 'mnth','holiday','weekday','workingday','weathersit']
X, y = df[feature_names].values, df['cnt'].values
pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]).fit(X, y)
print('学習しました')

## 1. 標準化係数を取り出して図にする

「モデルの中身を読む」第一歩です。パイプラインは標準化を含んでいるので、取り出した係数はそのまま比べられます。

### TODO①：係数の取り出しと棒グラフ

学習済みの `pipe` から線形回帰の係数を取り出し、変数名付きの横向き棒グラフにしてください。

### 深掘り：標準化係数は「他をそろえたうえでの効き」——そして早くも忍び寄る双子の影

いま取り出した係数を、影響の大きい順に並べるとこうなります（`coef_` の実行値）：**年 `yr` +1020**、**体感温度 `atemp` +582**、**季節 `season` +566**、**気温 `temp` +371**、**天気 `weathersit` −333**、風速 `windspeed` −198、湿度 `hum` −145……。標準化ずみのパイプラインなので、この数字はそのまま横並びで比べられます。棒グラフにすると、上位3本（年・体感温度・季節）が飛び抜けて長く、残りを引き離しているのが見えるはずです。

**「標準化係数」が意味すること**　これは「その変数が**ばらつき1つぶん（＝1標準偏差）大きくなると、利用台数が何台動くか**」を表します。単位（気温は0〜1、月は1〜12）のちがいを標準化で消したので、範囲の広い変数が不当に小さく見える／狭い変数が大きく見える、という不公平が起きません。だから「係数が大きい＝影響が大きい」を、ここではじめて胸を張って言えます。

**符号は「向き」、絶対値は「大きさ」**　解釈の第一歩は、符号が常識に合うかの確認です。`weathersit`（天気・悪いほど数字が大きい）が **−333** でマイナス、`windspeed` も `hum` もマイナス——悪天候・強風・蒸し暑さで自転車が敬遠される、という直感どおり。符号が常識と合っていれば、モデルが変な学習をしていないと安心できます。逆に、常識に反する符号が出たら、そこは共線性などを疑って読むべきサインです。

**係数は「他の変数をそろえたうえでの効き」**　ここが重回帰の係数の肝で、後の共線性の話にも直結します。重回帰の各係数は、単独の相関とはちがい、**「他の変数をすべて同じ値に固定したまま、その変数だけを1標準偏差動かしたときの効き」**（偏回帰係数）を表しています。第1回で見た単なる相関が「他をごちゃ混ぜにしたままの一緒の動き」だったのに対し、係数は「他を差し引いた、その変数固有の取り分」です。だからこそ交絡をある程度ならして読める——のですが、この「他を固定して1つだけ動かす」が**そもそも成り立たない**変数の組があると、話が崩れます。

**早くも引っかかる点**　一番効きそうな気温 `temp` が **+371** と、体感温度 `atemp`（+582）に負けています。気温と体感温度は、いわば同じ「暑さ・寒さ」を測った双子の列（第1回で相関 **0.99** を確認ずみ）。「他を固定して temp だけ動かす」といっても、temp が動けば atemp もほぼ一緒に動いてしまい、二つを別々に固定できません。この宙ぶらりんこそ、5節で正体を暴く**多重共線性**の最初の兆候です。いまは「上位3変数は年・気温まわり・季節」「temp が意外と小さいのが引っかかる」と押さえておけば十分です。

In [ ]:
# TODO: pipe から係数を取り出して、変数名付きの横向き棒グラフを描いてください

# 解答例①：係数の棒グラフ
coefs = pipe.named_steps['model'].coef_
order = np.argsort(coefs)
colors = ['#0066cc' if coefs[i] > 0 else '#e63946' for i in order]
plt.barh(np.array(feature_names)[order], coefs[order], color=colors, alpha=0.85)
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('標準化係数')
plt.tight_layout(); plt.show()


## 2. 答え合わせ：係数の一覧

答え合わせも兼ねて、係数を数字の一覧でも確認しておきます。この値を次のTODOで使います。

In [ ]:
coefs = pipe.named_steps['model'].coef_
coef_df = pd.DataFrame({'変数': feature_names, '標準化係数': coefs.round(0)})
print(coef_df.sort_values('標準化係数', key=abs, ascending=False).to_string(index=False))

## 3. 係数を日本語の文に翻訳する

数字のままでは人に伝わりません。定番の型——「◯◯がばらつき1つぶん大きいと、利用台数がおよそ△△台動く」——に当てはめて、日本語にします。

### TODO②：係数の翻訳（コメント記入式）

上の一覧を見て、下のコメントの `???` を自分の言葉で埋めてください（コメントを書き換えるだけでOK）。

### 深掘り：係数を日本語にするときの作法と、「非有意＝効果なし」ではないという落とし穴

**翻訳の型を、実データで**　標準化係数は「ばらつき1つぶん動くと、利用台数が何台動くか」でした。年 `yr` の係数 **+1020** をそのまま訳すと「年がばらつき1つぶん大きいと、およそ1020台多い」。ただし `yr` は 0（一年目）か 1（二年目）の2値で、その標準偏差はおよそ 0.5。つまり「一年目→二年目」という**実際の1段の変化は約2標準偏差ぶん**に相当し、効きは 1020×2 ≈ **およそ2000台**になります。だから「二年目は、サービスの普及を反映して実質およそ2000台多い」と訳すのが正確です。標準化係数を素の変化に訳し戻すときは、この「1標準偏差が現実の何段ぶんか」を一度はさむ、と覚えておくと安全です。天気 `weathersit`（−333）なら「天気の区分が1段悪くなるほど、利用は数百台ずつ減る」といった具合。数字を文にできて、はじめて「解釈した」と言えます。

**つまずき：係数が小さい／非有意＝効果なし、ではない**　ここが今回いちばんの落とし穴です。気温 `temp` の係数は **+371** と、いかにも控えめ。もし有意性の検定（係数がゼロと区別できるか）にかければ、`temp` も `atemp` も**「有意でない」と判定されかねません**。ですが、これを「気温は効いていない」と読むのは**重大な誤り**です。5節で見るとおり、双子の `atemp` を外せば `temp` の係数は **+943** へ跳ね上がる——気温まわりは明らかに強く効いています。効いていないのではなく、**双子で手柄を奪い合った結果、片方ずつの係数が小さく・不安定に見えているだけ**なのです。

**なぜそうなるのか（さわりだけ）**　共線性があると、個々の係数の「推定のぶれ幅（標準誤差）」が大きく膨らみます。ぶれ幅が大きいほど「ゼロと区別できない＝非有意」と出やすい。だから**非有意は「効果がない」ではなく「この1変数だけでは効きを切り分けられない」の合図**と読むのが正解です。判断を誤らないコツは、双子を**個別に**見ず、`temp+atemp` の**合計（グループ）として**「気温まわりは効く」と読むこと。個別係数の額面や有意・非有意に飛びつかない——これが共線性下の解釈の作法です。膨らむぶれ幅の正体は、次の5節で $(X^\top X)^{-1}$ と VIF を使って数値で突き止めます。

**もうひとつの落とし穴：係数は「関連」であって「因果」ではない**　翻訳のときに最も踏み外しやすいのがここです。「体感温度の係数が +582 と大きい」を、「**気温を上げれば利用が増える**」と読んではいけません。係数が語るのは、あくまで**データ上でどう一緒に動いていたか（関連）**であって、その変数をこちらが操作したら結果がどう変わるか（因果）ではありません。暖かい季節はお出かけも増える、といった背後の共通要因（交絡）が、気温と利用台数の関連を水増ししている可能性が残るからです。だから報告に書けるのは「**気温まわりの関連が強い**」まで。「気温を上げれば〜」と因果で言い切るには、交絡をそろえた別の慎重な検討が要ります。言い過ぎない勇気も、解釈の作法のうちです。

In [ ]:
# TODO: 係数の一覧を見て、??? の部分を自分の言葉で埋めてください

# 解答例②：翻訳（例）
print('年: 二年目は実質およそ2000台多い（普及の効果）')
print('天気: 天気が悪くなるほど利用台数は減る（係数がマイナス）')


## 4. 並べ替え重要度で裏を取る

係数は共線性でぐらつくことがあるので、別の原理の方法でも確かめます。変数をシャッフルして性能の落ち方を測るのが並べ替え重要度でした。

### TODO③：permutation_importance の実行と上位確認

並べ替え重要度を計算し、重要度の大きい順に変数を表示して、係数の図と顔ぶれを見比べてください。

### 深掘り：並べ替え重要度は「別の原理」だから裏取りになる——ただし双子には共通の死角

**なぜ別の道具で確かめるのか**　係数は強力ですが、いま見たとおり共線性でぐらつきます。そこで、**まったくちがう原理**の物差しで裏を取ります。並べ替え重要度は、ある変数の列だけを**シャッフルして情報を壊し**、予測性能がどれだけ落ちるかを測ります。大きく落ちれば「その変数は効いていた」、落ちなければ「もともと効いていなかった」。係数のように連立方程式を解くのではなく、**壊して性能の変化を見る**——原理がちがうからこそ、独立した裏取りになります。`n_repeats=10` はシャッフルの偶然をならすための繰り返しです。

**実データの並び**　この演習で計算すると、重要度の上位は **年 `yr` 0.56**、**体感温度 `atemp` 0.18**、**季節 `season` 0.17**、続いて `temp` 0.07、`weathersit` 0.06……。係数の図と**同じ顔ぶれ（年・気温まわり・季節）が上位**に来ました。ちがう原理で同じ結論が出ると、解釈の信頼はぐっと上がります。第5回で複数の指標を並べて見たのと同じ、「一つの数字を鵜呑みにしない」姿勢です。

**気温 `temp` だけが沈む理由**　ところが `temp` は、係数では4番手（+371）だったのに、重要度では `season` にも抜かれて小さく沈みます。理由は共線性そのもの。`temp` をシャッフルで壊しても、**双子の `atemp` がそっくりな情報を肩代わり**するので、モデルの性能はほとんど落ちません。だから「壊しても平気＝重要でない」と出てしまう。これも、係数の跳ね上がりと並ぶ、多重共線性のはっきりした指紋です。

**別の見方：一致は心強い、食い違いは手がかり**　2つの見方が**一致する部分**（年・季節）は自信をもって報告でき、**食い違う部分**（temp が重要度で沈む）は「共線性を疑え」の警告として使えます。食い違いはノイズではなく情報なのです。ただし一つ注意——双子の場合、**係数も重要度も同じ向きに `temp` を過小評価します**（どちらも「atemp が肩代わりする」ことに引きずられる）。つまりこの2つは、双子に関しては完全に独立な裏取りにはなりません。だからこそ、片方を外す5節の実験や、次の第7回でのブートストラップといった、**別角度の確認**が要るのです。

**重要度の数字は何の単位か**　並べ替え重要度の値（`yr` の 0.56 など）は、その列を壊したときに**決定係数 R² がどれだけ下がったか**を表します。`yr` を壊すと R² が 0.56 も落ちる＝それだけ予測を支えていた、ということ。絶対値そのものより、**変数どうしの大小関係**を読む道具だと思えば十分です。報告では「係数（向きと大きさ）」と「重要度（壊すと効くか）」を並べて示すと、『どちらへ効くか』と『どれだけ効くか』の両面から根拠を語れて説得力が増します。

In [ ]:
# TODO: 並べ替え重要度を計算し、重要度の大きい順に変数名と値を表示してください

# 解答例③：並べ替え重要度
r = permutation_importance(pipe, X, y, n_repeats=10, random_state=0)
for i in np.argsort(r.importances_mean)[::-1]:
    print(f'{feature_names[i]:12s}: {r.importances_mean[i]:.3f}')
# → 上位は yr, atemp, season。係数の図と同じ顔ぶれ。temp は共線性で重要度が沈む


## 5. 多重共線性を実験で確かめる

第1回で見つけた「そっくりな2列」temp と atemp（相関0.99）。片方を外すと、もう片方の係数はどう動くでしょうか。

### 深掘り：係数が入れ替わり暴れる幾何と、$(X^\top X)^{-1}$・VIF が測る「膨張」

いよいよ本題、多重共線性の正体です。上の実験の出力を、じっくり読み解きます。

**実験が示したこと**　全変数を入れたとき `temp` の係数は **+371**。ところが双子の `atemp` を外したとたん、**+943**——およそ2.5倍に跳ね上がります。逆に `temp` を外せば `atemp` が **+582 → +952** へ。片方を抜いただけで係数がこれほど動くのは、**額面どおり読んではいけない**という明確な警告です。一方で、注目すべきは**二つの合計** `temp + atemp` ≈ **+953** が、どちらを外しても外さなくてもほぼ動かないこと。つまりデータは、「**双子あわせた気温の効き（合計 ≈ 950）は強く決められる**」のに、「**その手柄を temp と atemp のどちらへどれだけ振るか**は決められない」のです。この「割り振りが決まらない」状態を、統計では**識別できない**と言います。

**幾何のイメージ（さわり）**　当てはまりの良さ（誤差）を係数の地形として描くと、`yr` のような他と無関係な変数の向きには**底が一点に決まるすり鉢**ができます。ところが双子の向きには、`合計 ≒ 950` を保ったまま「temp に多め／atemp に多め」と分け方を変えても誤差がほとんど変わらない、**底が平らに伸びた谷（尾根）**ができる。データが少し変われば、解はこの平らな谷底をスルスル横すべりします。だから**分け方（片方の係数）は暴れ、合計は動かない**。この平らな谷をブートストラップで揺らして可視化し、傾きを与えて底を一点に定める（＝解決する）話は、次の**第7回の演習**で正面から扱います。

**骨子：係数のぶれ幅は $(X^\top X)^{-1}$ の対角で決まる**　なぜ双子だけが暴れるのかは、係数の分散の式に骨組みが見えます。最小二乗の係数のばらつきは、ざっくり

$$
\mathrm{Var}(\hat{\beta}) \;=\; \sigma^2\,(X^\top X)^{-1}
$$

という形で表せます（$\sigma^2$ は予測の残差のばらつき、$X$ は説明変数の行列）。厳密な導出は追いません。押さえるのは一点——**係数 $j$ のぶれ幅は、この行列 $(X^\top X)^{-1}$ の $j$ 番目の対角成分に比例する**、ということです。そしてこの対角成分は、**変数 $j$ が他の変数たちからどれだけ言い当てられるか**で膨らみます。他と無関係な変数（`yr`）なら対角は小さく、係数は安定。双子のように他からほぼ完全に言い当てられる変数では、対角が跳ね上がり、係数が暴れるのです。

**VIF：膨張を1つの数字にした指標**　この「膨張の度合い」を変数ごとに1つの数字にしたのが **VIF（分散拡大係数）** です。定義は

$$
\mathrm{VIF}_j \;=\; \frac{1}{1 - R_j^2}
$$

で、$R_j^2$ は「**他のすべての変数で変数 $j$ を回帰したときの当てはまり**」。他からよく言い当てられる（$R_j^2$ が1に近い）ほど、VIF は無限大へ発散します。VIF が意味するのは、「**他と無相関だった理想の場合に比べ、その係数の分散が何倍に膨らんでいるか**」です。このデータで実際に計算すると、`temp` の VIF は **約63**、`atemp` は **約64**（10を超えたら黄信号、が目安）。裏を返せば $R_j^2 = 1 - 1/63 \approx$ **0.984**——温度まわりは他の変数（実質はもう片方の双子）から98%言い当てられる、ほぼ重複した列だということです。分散が63倍なら、標準誤差（ぶれ幅）は $\sqrt{63}\approx$ **8倍**。`temp` の係数が8倍も広い幅でぐらつくなら、+371 と +943 のあいだを行き来するのも当然です。ちなみに `yr` の VIF はほぼ **1.0**（膨張ゼロ）——だから年の係数は安心して読めるのでした。

VIF と先の行列は、じつは同じものです。標準化したデータでは、$(X^\top X)^{-1}$ の対角成分はちょうど $\mathrm{VIF}_j / n$ に等しく、実際このデータでも「$n \times$ 対角成分」を計算すると temp で **63.3** と、VIF に一致します。**「対角が膨らむ＝VIF が大きい＝係数が暴れる」は、ぜんぶ一つの現象の言い換え**なのです。（別の見方をすれば、双子があると説明変数の相関行列の固有値の一つが 0 に近づき、その方向＝平らな谷になっている、とも言えます。）

**つまずきの総まとめ**　だから共線性の下では——(1) **個々の係数を額面どおり読まない**（+371 は「効きが弱い」ではなく「双子と分け合った取り分」）。(2) **小さい係数・非有意を「効果なし」と早合点しない**（ぶれ幅が膨らんで有意に見えないだけ）。(3) 双子は**合計（グループ）として** 「気温まわりは強く効く」と読む。(4) 予測の精度自体はほとんど落ちない（合計は安定なので）——**精度は出るのに解釈だけが信用できない**、これが多重共線性のいちばん厄介な顔です。

**次への橋渡し**　対策は大きく2つ。素朴には**双子の片方を落とす**（合計が安定なので予測はほぼ変わらない）。もう一つが、平らな谷にわずかな傾きを与えて底を一点に定める**正則化**です。係数を安定させて解釈を守るこの道具——リッジ回帰とラッソ回帰——を、**第7回の演習**で自分の手で動かし、ここで暴いた双子に決着をつけます。この回は問題を**暴いて診断する**ところまで。分散の式 $\sigma^2 (X^\top X)^{-1}$ や VIF の厳密な導出に踏み込みたい人は、**詳しくは MVA『重回帰』回**へどうぞ。

In [ ]:
def temp_coef(cols):
    p = Pipeline([('s', StandardScaler()), ('m', LinearRegression())]).fit(df[cols].values, y)
    return p.named_steps['m'].coef_[cols.index('temp')]

print(f"全変数のとき  temp の係数: {temp_coef(feature_names):+.0f}")
print(f"atemp を外すと temp の係数: {temp_coef([c for c in feature_names if c!='atemp']):+.0f}")
print('→ 双子の atemp を外すと、temp の係数が跳ね上がる（多重共線性）')

## 目標に答えられたか

- 今日の目標は「何が利用台数を動かすのか、根拠をもって説明したい」でした
- TODO①の図で、影響が大きい上位3変数は何でしたか？ 符号（向き）は常識と合っていましたか？
- TODO③の並べ替え重要度と、TODO①の係数で、上位の顔ぶれは一致しましたか？ 一致すると何が言えるでしょう？
- 5節の実験で、atempを外すとtempの係数はどう動きましたか？ この2つの係数を額面どおり読んでよいでしょうか？
- この結果を「性能・解釈・限界」の三点セットで、1〜2文の報告にまとめてみましょう。

## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

1節では、標準化を含むパイプライン `pipe` の係数（標準化係数）を棒グラフにしました。今度は標準化を外します。
`LinearRegression()` を生の `X` にそのまま学習させ、その係数（生係数）を、TODO①と同じ変数名付きの横向き棒グラフにしてください。


In [ ]:
raw = LinearRegression().fit(X, y)          # 標準化なしで学習
raw_coefs = raw.coef_
order = np.argsort(raw_coefs)
colors = ['#0066cc' if raw_coefs[i] > 0 else '#e63946' for i in order]
plt.barh(np.array(feature_names)[order], raw_coefs[order], color=colors, alpha=0.85)
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('生の係数（標準化なし）')
plt.tight_layout(); plt.show()
# 実行値: atemp +3573, windspeed -2558, yr +2041, temp +2029, hum -1019, ... 標準化係数と順位が入れ替わる


<details><summary>詰まったら</summary>

TODO①のコードで、`pipe.named_steps['model']` の代わりに `LinearRegression().fit(X, y)` を使うだけです。その `coef_` を棒グラフにします。

</details>


### 応用②（判断）

応用①の生係数で、**絶対値が最大**の変数はどれですか。変数名（`feature_names` にある英語の列名）で答えてください。


In [ ]:
j = int(np.argmax(np.abs(raw_coefs)))
print(f'生係数で絶対値が最大: {feature_names[j]}（{raw_coefs[j]:+.0f}）')
print('答え:', feature_names[j])


<details><summary>詰まったら</summary>

`np.argmax(np.abs(生係数))` で位置が分かります。`feature_names[その位置]` が変数名です。

</details>


### 応用③（解釈）

標準化係数（1節）と生係数（応用①）とでは、影響の大きい変数の順位が入れ替わります。
なぜ順位が入れ替わるのか（変数の単位や取りうる範囲の違いに触れて）、そして「何が利用台数を動かすか」の報告にはどちらの係数を使うべきかを、自転車シェアの運営担当者に向けて3行程度で書いてください。


**模範例**

生係数は「その変数が 1 だけ増えたときの台数」ですが、体感温度 atemp や風速 windspeed は 0〜1 の範囲で、「1 増える」ことは現実に起こりません。範囲の狭い変数ほど係数が大きく見え、風速（生係数 −2558、範囲 0.02〜0.51）が 2 位に躍り出るのはそのためです。

「何が利用台数を動かすか」を横並びで比べる報告には、全変数を「ばらつき1つぶん」にそろえた標準化係数（年 +1020、体感温度 +582、季節 +566）を使います。

生係数は「天気の区分が 1 段悪くなると約 600 台減る（生係数 −611）」のように、1 増えることが現実に起こる変数で具体的な変化幅を伝えるときに使い分けます。ただし temp と atemp の双子はどちらの係数でも取り分が決まらないので、報告では「気温まわり」とまとめ、決着は第7回でつけます。


<details><summary>詰まったら</summary>

`df[feature_names].describe()` で各変数の最小値・最大値を見比べてください。「1増える」が現実に起こりうる変数と、起こりえない変数があります。

</details>


## 発展（任意）

### SHAP で「この1日」の予測を変数ごとに分解する

係数は「平均的な効き方」を表します。ばらつき1つぶんで何台、という話でした。ところが現場で聞かれるのは「**この日**の予測はなぜ高いのか」です。係数だけでは、その日の値を一つひとつ掛け合わせて足す作業を自分でやらなければ答えられません。

SHAP（シャープリー加法的説明）は、1件ごとの予測を「全体の平均予測 ＋ 各変数の寄与」に**分配**する枠組みです。線形回帰なら寄与は「係数 × その日の値のずれ（平均から標準偏差の何倍か）」で、係数の話とぴったりつながります。

そして同じ枠組みが、ランダムフォレストやニューラルネットのような、係数を持たないモデルにも使えます。「1件を説明する」道具として、一度手を動かしておく価値があります。


In [ ]:
# Colab には入っていないので、なければインストールする
import importlib.util
if importlib.util.find_spec('shap') is None:
    %pip install -q shap


In [ ]:
import shap

X_s = pipe.named_steps['scaler'].transform(X)                   # 標準化ずみの X（モデルが実際に見ている値）
masker = shap.maskers.Independent(X_s, max_samples=len(X_s))   # 全731日を「基準」にする（既定だと100日の標本になる）
explainer = shap.Explainer(pipe.named_steps['model'], masker)  # 線形モデルなので LinearExplainer が選ばれる
sv = explainer(X_s)                                             # 全日ぶんの SHAP 値（731行 × 11変数）

i = int(np.argmax(y))                                           # 利用台数が最大だった日
print(f"{df['dteday'][i]}：実測 {y[i]} 台、予測 {pipe.predict(X[[i]])[0]:.0f} 台、全日の平均予測 {sv.base_values[i]:.0f} 台")
for j in np.argsort(-np.abs(sv.values[i])):
    print(f'{feature_names[j]:12s} 値={X[i, j]:7.3f}  寄与={sv.values[i, j]:+8.1f} 台')


In [ ]:
# 1日ぶんの寄与を、平均予測から積み上げる滝グラフ（waterfall）にする
one = shap.Explanation(values=sv.values[i], base_values=sv.base_values[i],
                       data=X[i], feature_names=feature_names)   # data に生の値を渡すと、図の左に元の値が出る
shap.plots.waterfall(one, max_display=12)


**読み方**　左下の $E[f(X)]$ = 4504 台が「基準」、全日の平均予測です（線形回帰では実測の平均と一致します）。そこから変数ごとの寄与を足し引きして、右上の $f(x)$ = 6675 台、この日（2012-09-15、土曜・`season`=3・晴れ・二年目）の予測にたどり着きます。赤が押し上げ、青が押し下げです。

最大の寄与は **年 `yr` の +1019 台**。二年目（`yr`=1）はずれ（平均から標準偏差の何倍か）がほぼ 1 なので、標準化係数 +1020 がほぼそのまま寄与になります。次が体感温度 `atemp` の +398 台（係数 +582 × ずれ 0.685）、季節 `season` の +257 台。temp と atemp の寄与の分け方は係数の取り分と同じく決まらないので、ここも合計（+628 台）で読みます。

天気 `weathersit` は係数がマイナス（−333）なのに、この日の寄与は **+242 台** と赤です。晴れ（=1）は平均より良い天気、つまりマイナス側へのずれなので、「マイナス × マイナス」で押し上げに転じるからです。係数は「向き」を、SHAP はその日の値と掛け合わせた「実際の押し引き」を語ります。

なお、この日の実測は 8714 台で、予測 6675 台を 2000 台以上も上回ります。滝グラフが説明するのはあくまで**モデルの予測**であって、実測ではありません。予測が外れた理由までは教えてくれない点に注意してください。シャープリー値の厳密な定義（寄与を公平に分配するための公理）は、ここでは追いません。

試すなら、`i = int(np.argmin(y))`（利用が最も少なかった 2012-10-29、ハリケーンの日）に変えてみてください。天気 `weathersit`=3 の寄与が −980 台と、最大の押し下げ要因に変わります。
